In [6]:
# Install required libraries
!pip install ultralytics opencv-python-headless

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from google.colab import files
from google.colab.patches import cv2_imshow

print("All libraries installed successfully!")

All libraries installed successfully!


In [7]:
print("Click 'Choose Files' to upload your padel video")

uploaded = files.upload()

video_filename = list(uploaded.keys())[0]

print(f"Uploaded Video: {video_filename}")

Click 'Choose Files' to upload your padel video


Saving input_sample_video.mp4 to input_sample_video (1).mp4
Uploaded Video: input_sample_video (1).mp4


In [8]:
model = YOLO("yolov8n.pt")

print("YOLO Model Loaded Successfully!")


YOLO Model Loaded Successfully!


In [11]:
cap = cv2.VideoCapture(video_filename)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video Resolution : {width}x{height}")
print(f"FPS              : {fps}")
print(f"Total Frames     : {total_frames}")


out = cv2.VideoWriter(
    "output_padel.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

#variables
ball_positions = []

forehand_count = 0
backhand_count = 0
smash_count = 0

prev_direction = None
frame_num = 0

shot_data = []
#  Process Video
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_num += 1
    # YOLO Detection
    # 0  = person
    # 32 = sports ball
    results = model(
        frame,
        classes=[0, 32],
        conf=0.30,
        verbose=False
    )

    ball_x = None
    ball_y = None
    # Detection Parsing


    for r in results:

        boxes = r.boxes

        if boxes is None:
            continue

        for box in boxes:

            cls = int(box.cls[0])

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # PLAYER DETECTION


            if cls == 0:

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (255, 0, 0),
                    2
                )

                cv2.putText(
                    frame,
                    "PLAYER",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 0, 0),
                    2
                )


            # BALL DETECTION


            elif cls == 32:

                ball_x = (x1 + x2) // 2
                ball_y = (y1 + y2) // 2

                cv2.circle(
                    frame,
                    (ball_x, ball_y),
                    10,
                    (0, 255, 0),
                    3
                )

                cv2.putText(
                    frame,
                    "BALL",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )


    # Ball Tracking + Shot Classification

    if ball_x is not None:

        ball_positions.append((ball_x, ball_y))

        # Draw trajectory
        for i in range(1, len(ball_positions)):

            cv2.line(
                frame,
                ball_positions[i - 1],
                ball_positions[i],
                (0, 255, 255),
                2
            )

        # Keep last 30 positions
        if len(ball_positions) > 30:
            ball_positions.pop(0)

        # Shot Detection Logic


        if len(ball_positions) >= 2:

            prev_x, prev_y = ball_positions[-2]
            curr_x, curr_y = ball_positions[-1]

            dx = curr_x - prev_x
            dy = curr_y - prev_y

            # Direction detection
            if dx > 5:
                current_direction = "right"

            elif dx < -5:
                current_direction = "left"

            else:
                current_direction = prev_direction

            shot_type = None


            # Forehand / Backhand


            if prev_direction is not None:

                if current_direction != prev_direction:

                    if current_direction == "right":

                        forehand_count += 1
                        shot_type = "Forehand"

                    elif current_direction == "left":

                        backhand_count += 1
                        shot_type = "Backhand"

            # Smash Detection


            if abs(dy) > 25:

                smash_count += 1
                shot_type = "Smash"

            prev_direction = current_direction

            # Save Shot Data


            if shot_type is not None:

                timestamp = round(frame_num / fps, 2)

                shot_data.append({
                    "frame": frame_num,
                    "timestamp": timestamp,
                    "shot_type": shot_type
                })

                print(f"{shot_type} detected at frame {frame_num}")

                # Display current shot
                cv2.putText(
                    frame,
                    f"{shot_type} Shot",
                    (50, 150),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 0, 255),
                    3
                )
    # Dashboard Overlay


    cv2.rectangle(
        frame,
        (10, 10),
        (350, 140),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        frame,
        f"Forehand : {forehand_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 100),
        2
    )

    cv2.putText(
        frame,
        f"Backhand : {backhand_count}",
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 200, 0),
        2
    )

    cv2.putText(
        frame,
        f"Smash : {smash_count}",
        (20, 110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 100, 255),
        2
    )
    # Write Output Frame


    out.write(frame)


    # Progress


    if frame_num % 50 == 0:

        print(f"Processing Frame {frame_num}/{total_frames}")

# STEP 9 : Save CSV


df = pd.DataFrame(shot_data)

df.to_csv(
    "padel_shot_analysis.csv",
    index=False
)


# STEP 10 : Release Resources


cap.release()
out.release()

print("\nProcessing Completed Successfully!")
print("Output Video Saved!")
print("CSV File Saved!")

print("\nFinal Statistics")
print(f"Forehand : {forehand_count}")
print(f"Backhand : {backhand_count}")
print(f"Smash    : {smash_count}")

# STEP 11 : Download Files


files.download("output_padel.mp4")
files.download("padel_shot_analysis.csv")

Video Resolution : 1920x1080
FPS              : 25
Total Frames     : 8125
Processing Frame 50/8125
Processing Frame 100/8125
Processing Frame 150/8125
Processing Frame 200/8125
Processing Frame 250/8125
Smash detected at frame 285
Smash detected at frame 298
Processing Frame 300/8125
Smash detected at frame 336
Processing Frame 350/8125
Smash detected at frame 382
Processing Frame 400/8125
Processing Frame 450/8125
Processing Frame 500/8125
Smash detected at frame 542
Processing Frame 550/8125
Smash detected at frame 570
Forehand detected at frame 585
Processing Frame 600/8125
Processing Frame 650/8125
Processing Frame 700/8125
Processing Frame 750/8125
Processing Frame 800/8125
Smash detected at frame 823
Processing Frame 850/8125
Processing Frame 900/8125
Processing Frame 950/8125
Processing Frame 1000/8125
Processing Frame 1050/8125
Processing Frame 1100/8125
Processing Frame 1150/8125
Processing Frame 1200/8125
Processing Frame 1250/8125
Processing Frame 1300/8125
Processing Frame

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>